In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

# GRPO Training – TeenyTinyLlama

Three sections, run in order:

1. **SFT Warm-up** *(optional)* — supervised fine-tuning teaches the entity-extraction output format → **demo after training**  
2. **GRPO Training** — RL with the NER reward function  
3. **Evaluation** — reward-based F1 scores and a model demo  

> **Local run** (these defaults): `TTLlama-160m` on MPS.  
> **Cluster run**: swap `MODEL_NAME`, scale up batch sizes, point `SFT_ADAPTER_PATH` to a finished SFT checkpoint.

In [2]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME = "nicholasKluge/TeenyTinyLlama-160m"   # swap to TeenyTinyLlama-460m for more capacity
DEVICE     = None   # None = auto-detect (MPS > CUDA > CPU)

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_PRESET = "full_attention"   # see src/peft_configs/lora.py for all presets

# ── Checkpoints ────────────────────────────────────────────────────────────
RUN_NAME            = "ttllama_160m"
SFT_CHECKPOINT_DIR  = f"../outputs/checkpoints/{RUN_NAME}_sft"
GRPO_CHECKPOINT_DIR = f"../outputs/checkpoints/{RUN_NAME}_grpo"

# Warm-start GRPO from a finished SFT run.
# Set to None to start GRPO from base model weights (fine for pipeline testing).
# SFT_ADAPTER_PATH = None
SFT_ADAPTER_PATH = f"{SFT_CHECKPOINT_DIR}/checkpoint-979"

# ── SFT hypers (local-friendly defaults) ───────────────────────────────────
SFT_EPOCHS     = 3
SFT_BATCH_SIZE = 2
SFT_GRAD_ACCUM = 4

# ── GRPO hypers (local-friendly defaults) ──────────────────────────────────
GRPO_EPOCHS          = 1
GRPO_BATCH_SIZE      = 1
GRPO_GRAD_ACCUM      = 4
GRPO_NUM_GENERATIONS = 4     # completions per prompt; lower = less memory
GRPO_MAX_PROMPT_LEN  = 512
GRPO_MAX_COMP_LEN    = 128

# ── Evaluation ─────────────────────────────────────────────────────────────
EVAL_SAMPLE_SIZE = 50   # set to None to run on the full test split

In [3]:
from src.models.ttllama import TTLlama

t = TTLlama(model_name=MODEL_NAME, device=DEVICE)
print(t.model)

/Users/mstauffer/Documents/unb/research/decoder-lener-iob/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:src.models.ttllama:Using device: mps
INFO:src.models.ttllama:Tokenizer loaded — pad_token_id=3, eos_token_id=2
`torch_dtype` is deprecated! Use `dtype` instead!


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 768, padding_idx=3)
    (layers): ModuleList(
      (0-11): 12 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=False)
          (k_proj): Linear(in_features=768, out_features=768, bias=False)
          (v_proj): Linear(in_features=768, out_features=768, bias=False)
          (o_proj): Linear(in_features=768, out_features=768, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=768, out_features=3072, bias=False)
          (up_proj): Linear(in_features=768, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=768, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((768,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((768,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((768,), eps=1e-05)
    (r

In [4]:
print(f"EOS  : '{t.tokenizer.eos_token}'  (id={t.tokenizer.eos_token_id})")
print(f"PAD  : '{t.tokenizer.pad_token}'  (id={t.tokenizer.pad_token_id})")
print(f"BOS  : '{t.tokenizer.bos_token}'  (id={t.tokenizer.bos_token_id})")
assert t.tokenizer.pad_token_id != t.tokenizer.eos_token_id, "pad == eos — fix before training!"
print("pad != eos: OK")

EOS  : '</s>'  (id=2)
PAD  : '<pad>'  (id=3)
BOS  : '<s>'  (id=1)
pad != eos: OK


---

## 1 · SFT Warm-up (Optional)

Supervised fine-tuning teaches the model the entity-extraction output format (`TAG: entity; TAG: entity`) before GRPO.  
**Skip this section** if you have a pre-trained SFT checkpoint set in `SFT_ADAPTER_PATH`, or if you only want to test the GRPO pipeline mechanics.

Training format: `prompt + ground_truth + EOS` — loss is computed on the full sequence.

In [5]:
from src.datasets.lener import LenerDataset

l = LenerDataset(tokenizer=t.tokenizer)
raw = l.load_dataset(format="grpo")   # returns {prompt, ground_truth} columns

# Build full supervised sequence: prompt + answer + EOS
sft_data = raw.map(
    lambda ex: {"text": ex["prompt"] + ex["ground_truth"] + t.tokenizer.eos_token},
    remove_columns=raw["train"].column_names,
)

print(sft_data)
print("\nSample (truncated to 500 chars):")
print(sft_data["train"][0]["text"][:500])

INFO:src.datasets.lener:Loading dataset: peluz/lener_br
Using the latest cached version of the dataset since peluz/lener_br couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'lener_br' at /Users/mstauffer/.cache/huggingface/datasets/peluz___lener_br/lener_br/1.0.0/4a8c97e6813b5c2d85a50faf0a3e6c24ea82f4a9044e6e9e8b24997d27399382 (last modified on Sun May 24 17:03:00 2026).
INFO:src.datasets.lener:Dataset loaded with splits: ['train', 'validation', 'test']
Map: 100%|██████████| 1390/1390 [00:00<00:00, 58851.08 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['text'],
        num_rows: 1390
    })
})

Sample (truncated to 500 chars):
Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.
- PESSOA: Designa entidades que são nomes de pessoas físicas.
- TEMPO: Marca entidades que expressam informações temporais, como datas, horários, períodos, etc.
- LOCAL: Indica entidades que representam lugares geográficos, como cidades, países


In [6]:
from trl import SFTTrainer, SFTConfig
from src.peft_configs.lora import LoraAdapter

lora_sft = LoraAdapter(model=t.model, lora_preset=LORA_PRESET)
sft_model = lora_sft.apply_lora()

sft_args = SFTConfig(
    output_dir=SFT_CHECKPOINT_DIR,
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    save_strategy="epoch",
    bf16=(t.device != "cpu"),
    fp16=False,
    dataset_text_field="text",
    max_length=1024,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_args,
    train_dataset=sft_data["train"],
    processing_class=t.tokenizer,
)

sft_trainer.train()

INFO:root:Applying LoRA with preset: full_attention
INFO:root:LoRA configurations: {'r': 32, 'lora_alpha': 64, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'], 'lora_dropout': 0.1, 'bias': 'none', 'task_type': <TaskType.CAUSAL_LM: 'CAUSAL_LM'>}


trainable params: 2,359,296 || all params: 164,776,704 || trainable%: 1.4318


Truncating train dataset: 100%|██████████| 7828/7828 [00:00<00:00, 584789.59 examples/s]
/Users/mstauffer/Documents/unb/research/decoder-lener-iob/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
20,3.047700
40,2.971500
60,2.661500
80,2.180700
100,1.399400
120,0.902700
140,0.873200
160,0.707500
180,0.677000
200,0.719500


/Users/mstauffer/Documents/unb/research/decoder-lener-iob/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: defc89c0-e5d4-49fa-b09c-aab1ccbf27ef)')' thrown while requesting HEAD https://huggingface.co/nicholasKluge/TeenyTinyLlama-160m/resolve/main/config.json
Retrying in 1s [Retry 1/5].
/Users/mstauffer/Documents/unb/research/decoder-lener-iob/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 4c91a884-7139-452b-9cb6-c9ac65

TrainOutput(global_step=2937, training_loss=0.6076689924071912, metrics={'train_runtime': 17778.0668, 'train_samples_per_second': 1.321, 'train_steps_per_second': 0.165, 'total_flos': 5102073541813248.0, 'train_loss': 0.6076689924071912, 'entropy': 0.40583066484241775, 'num_tokens': 5368194.0, 'mean_token_accuracy': 0.9207055640943123, 'epoch': 3.0})

SFT checkpoint saved to `SFT_CHECKPOINT_DIR`.  
Update `SFT_ADAPTER_PATH` in the CONFIG cell to use it as a GRPO warm-start.

Run the cell below to verify the model learned the entity-extraction format before committing to GRPO training.  
If the output still looks like raw text continuation, run more SFT epochs first.

In [9]:
# ── Seqeval evaluation utilities ────────────────────────────────────────────
# Adapted from src/scripts/eval_seqeval3.py for the entity-extraction format
# used by SFT and GRPO ("Resposta:\n" header, semicolon-separated entities).
# Entities are parsed → merged → aligned back to original tokens → BIO scored.

import re
import sys
import string
import torch
import nltk
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from seqeval.metrics import classification_report, f1_score as seqeval_f1

_KNOWN_LABELS = ["LEGISLACAO", "JURISPRUDENCIA", "TEMPO", "PESSOA", "ORGANIZACAO", "LOCAL"]


def _parse_entities(text):
    """Parse 'TAG: entity; ...' output, preserving original case for token matching."""
    if "Resposta:" in text:
        text = text.split("Resposta:")[-1]
    pattern = r"(" + "|".join(_KNOWN_LABELS) + r"):\s*([^;\n]+)"
    matches = re.findall(pattern, text, re.IGNORECASE)
    return [(tag.strip().upper(), val.strip()) for tag, val in matches]


def _preprocess_entity_text(text):
    processed = text.split("\n", 1)[0]
    while processed and processed[-1] in string.punctuation:
        processed = processed[:-1]
    return processed.strip()


def _find_prefix_before_match(text):
    if not text:
        return text
    patterns = {label[:i] for label in _KNOWN_LABELS for i in range(3, len(label) + 1)}
    first_idx = sys.maxsize
    for pat in patterns:
        idx = text.find(pat)
        if idx != -1 and idx < first_idx:
            first_idx = idx
    return text[:first_idx] if first_idx != sys.maxsize else text


def _filter_and_merge_entities(entities):
    if not entities:
        return []
    preprocessed = list({(lbl, _preprocess_entity_text(txt)) for lbl, txt in entities})
    if len(preprocessed) <= 1:
        return sorted(preprocessed)
    final = [
        (la, ta) for i, (la, ta) in enumerate(preprocessed)
        if not any(
            la == lb and ta != tb and ta in tb
            for j, (lb, tb) in enumerate(preprocessed) if i != j
        )
    ]
    return sorted(final)


def _match_entities_to_tokens(tokens, filtered_entities):
    bio_tags = ["O"] * len(tokens)
    if not filtered_entities:
        return bio_tags
    tokenized = []
    for label, ent_text in filtered_entities:
        cleaned = _find_prefix_before_match(ent_text).rstrip(string.punctuation + string.whitespace)
        ent_tokens = word_tokenize(cleaned)
        if ent_tokens:
            tokenized.append((label, ent_tokens))
    tokenized.sort(key=lambda x: len(x[1]), reverse=True)
    n = len(tokens)
    for label, ent_tokens in tokenized:
        ne = len(ent_tokens)
        for i in range(n - ne + 1):
            if tokens[i:i + ne] == ent_tokens:
                if all(bio_tags[i + j] == "O" for j in range(ne)):
                    bio_tags[i] = f"B-{label}"
                    for j in range(1, ne):
                        bio_tags[i + j] = f"I-{label}"
    return bio_tags


def evaluate_seqeval(eval_model, split, tokenizer, device, label_names, sample_size=None):
    """
    Evaluates a model on a dataset split using seqeval F1.

    Generates completions in entity-extraction format, converts entities to BIO tags
    via token alignment, and scores with seqeval.
    Returns (classification_report_str, macro_f1_float).
    """
    if sample_size:
        split = split.select(range(sample_size))

    y_true, y_pred = [], []
    eval_model.eval()

    with torch.no_grad():
        for example in split:
            gold_tags = [label_names[g] for g in example["ner_tags"]]

            inputs = tokenizer(
                example["prompt"],
                return_tensors="pt",
                truncation=True,
                max_length=GRPO_MAX_PROMPT_LEN,
            ).to(device)

            outputs = eval_model.generate(
                **inputs,
                max_new_tokens=GRPO_MAX_COMP_LEN,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            generated = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True,
            )

            entities  = _filter_and_merge_entities(_parse_entities(generated))
            pred_tags = _match_entities_to_tokens(example["tokens"], entities)

            y_true.append(gold_tags)
            y_pred.append(pred_tags)

    report = classification_report(y_true, y_pred)
    f1     = seqeval_f1(y_true, y_pred)
    return report, f1

In [10]:
_label_names = raw["train"].features["ner_tags"].feature.names

print(f"Evaluating SFT model on {EVAL_SAMPLE_SIZE or 'all'} test examples...")
sft_report, sft_f1 = evaluate_seqeval(
    eval_model=sft_model,
    split=raw["test"],
    tokenizer=t.tokenizer,
    device=t.device,
    label_names=_label_names,
    sample_size=EVAL_SAMPLE_SIZE,
)

print(f"\n[SFT] seqeval F1 : {sft_f1:.4f}")
print(f"\n[SFT] Classification Report:\n{sft_report}")

Evaluating SFT model on 50 test examples...

[SFT] seqeval F1 : 0.1243

[SFT] Classification Report:
                precision    recall  f1-score   support

JURISPRUDENCIA       0.00      0.00      0.00        15
    LEGISLACAO       0.00      0.00      0.00        23
         LOCAL       0.00      0.00      0.00         3
   ORGANIZACAO       0.80      0.06      0.11        67
        PESSOA       0.75      0.08      0.15        37
         TEMPO       0.80      0.40      0.53        10

     micro avg       0.50      0.07      0.12       155
     macro avg       0.39      0.09      0.13       155
  weighted avg       0.58      0.07      0.12       155



In [11]:
from src.scripts.utils import predict_entities_batch

SFT_DEMO_TEXTS = [
    "Tratando-se de ação indenizatória ajuizada por pessoa incapaz , é obrigatória a intervenção do Ministério Público na condição de fiscal da ordem jurídica .",
    "O ministério público acatou a decisão do STF e pediu a suspensão do processo contra o ex-presidente Lula , com base na Lei 17.681/2017 .",
]

extra_tokens = [
    "PESSOA", "ORGANIZACAO", "LOCAL", "TEMPO",
    "LEGISLACAO", "JURISPRUDENCIA", ":", ";", "\n", "-",
    t.tokenizer.eos_token,
]

results = predict_entities_batch(SFT_DEMO_TEXTS, sft_model, t.tokenizer, extra_tokens=extra_tokens)

print("── SFT model demo ──")
for text, result in zip(SFT_DEMO_TEXTS, results):
    resposta_idx = result.find("Resposta:")
    answer = result[resposta_idx:] if resposta_idx != -1 else result
    print(f"INPUT : {text}")
    print(f"OUTPUT: {answer.strip()}")
    print()

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


── SFT model demo ──
INPUT : Tratando-se de ação indenizatória ajuizada por pessoa incapaz , é obrigatória a intervenção do Ministério Público na condição de fiscal da ordem jurídica .
OUTPUT: Resposta:
LEGISLACAO

INPUT : O ministério público acatou a decisão do STF e pediu a suspensão do processo contra o ex-presidente Lula , com base na Lei 17.681/2017 .
OUTPUT: Resposta:
LEGISLACAO : Lei 17.681/2017



---

## 2 · GRPO Training

Loads the model fresh (with optional SFT warm-start), applies a new LoRA adapter, and runs GRPO with the NER reward function.

Reward function (`ner_reward_func`) returns:

| Condition | Reward |
|---|---|
| Perfect match or correctly empty | `1.0` |
| Partial match | F1 ∈ `(0, 1)` |
| Predicted nothing, ground truth non-empty | `-0.2` |
| Hallucinated entities on empty ground truth | `-1.0` |

In [14]:
import gc
import torch
from src.models.ttllama import TTLlama

# Reload a clean base model (discards any SFT LoRA weights from above).
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
elif torch.cuda.is_available():
    torch.cuda.empty_cache()

t = TTLlama(model_name=MODEL_NAME, device=DEVICE)

if SFT_ADAPTER_PATH:
    from peft import PeftModel
    t.model = PeftModel.from_pretrained(t.model, SFT_ADAPTER_PATH)
    t.model.config.pad_token_id = t.tokenizer.pad_token_id
    print(f"Loaded SFT adapter from: {SFT_ADAPTER_PATH}")
else:
    print("No SFT warm-start — GRPO runs from base model weights.")

INFO:src.models.ttllama:Using device: mps
INFO:src.models.ttllama:Tokenizer loaded — pad_token_id=3, eos_token_id=2


Loaded SFT adapter from: ../outputs/checkpoints/ttllama_160m_sft/checkpoint-979


In [15]:
from src.datasets.lener import LenerDataset

l = LenerDataset(tokenizer=t.tokenizer)
grpo_data = l.load_dataset(format="grpo")
print(grpo_data)

INFO:src.datasets.lener:Loading dataset: peluz/lener_br
Using the latest cached version of the dataset since peluz/lener_br couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'lener_br' at /Users/mstauffer/.cache/huggingface/datasets/peluz___lener_br/lener_br/1.0.0/4a8c97e6813b5c2d85a50faf0a3e6c24ea82f4a9044e6e9e8b24997d27399382 (last modified on Sat May 23 23:27:29 2026).
INFO:src.datasets.lener:Dataset loaded with splits: ['train', 'validation', 'test']


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'prompt', 'ground_truth'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'prompt', 'ground_truth'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'prompt', 'ground_truth'],
        num_rows: 1390
    })
})


In [16]:
sample = grpo_data["train"][0]
print("── PROMPT (first 400 chars) ──")
print(sample["prompt"][:400])
print("\n── GROUND TRUTH ──")
print(sample["ground_truth"])

── PROMPT (first 400 chars) ──
Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.
- PESSOA: Designa entidades que são nomes de pessoas físicas.
- TEMPO: Marca entidades que expressam informações temporais, como datas, horários

── GROUND TRUTH ──
ORGANIZACAO: MINISTÉRIO PÚBLICO


In [17]:
from src.peft_configs.lora import LoraAdapter

lora = LoraAdapter(model=t.model, lora_preset=LORA_PRESET)
model = lora.apply_lora()

INFO:root:Applying LoRA with preset: full_attention
INFO:root:LoRA configurations: {'r': 32, 'lora_alpha': 64, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'], 'lora_dropout': 0.1, 'bias': 'none', 'task_type': <TaskType.CAUSAL_LM: 'CAUSAL_LM'>}


trainable params: 2,359,296 || all params: 164,776,704 || trainable%: 1.4318


/Users/mstauffer/Documents/unb/research/decoder-lener-iob/venv/lib/python3.13/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/mstauffer/Documents/unb/research/decoder-lener-iob/venv/lib/python3.13/site-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
from trl import GRPOTrainer, GRPOConfig
from src.rewards.ner_reward import ner_reward_func

training_args = GRPOConfig(
    output_dir=GRPO_CHECKPOINT_DIR,
    num_train_epochs=GRPO_EPOCHS,
    per_device_train_batch_size=GRPO_BATCH_SIZE,
    gradient_accumulation_steps=GRPO_GRAD_ACCUM,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    learning_rate=5e-6,         # lower than SFT; we refine, not re-learn
    bf16=(t.device != "cpu"),
    fp16=False,
    report_to="none",
    # GRPO-specific ─────────────────────────────────────────────────────────
    num_generations=GRPO_NUM_GENERATIONS,   # completions per prompt
    max_prompt_length=GRPO_MAX_PROMPT_LEN,
    max_completion_length=GRPO_MAX_COMP_LEN,
    beta=0.04,                  # KL penalty; keep low if model already knows the format
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=grpo_data["train"],
    processing_class=t.tokenizer,
    reward_funcs=ner_reward_func,
)

trainer.train()

---

## 3 · Evaluation

### 3a — seqeval F1 on the test set

Runs greedy generation on a sample of the test split, converts entity-extraction outputs to BIO tags
via token alignment, and reports per-entity-type F1 with seqeval.

These scores are directly comparable to the SFT baseline computed in Section 1 — the delta between
`sft_f1` and `grpo_f1` is the measured contribution of GRPO on top of the SFT warm-start.

### 3b — Model demo

Pass any Portuguese legal text to see the GRPO model's entity extraction.

In [ ]:
print(f"Evaluating GRPO model on {EVAL_SAMPLE_SIZE or 'all'} test examples...")
grpo_report, grpo_f1 = evaluate_seqeval(
    eval_model=model,
    split=grpo_data["test"],
    tokenizer=t.tokenizer,
    device=t.device,
    label_names=grpo_data["train"].features["ner_tags"].feature.names,
    sample_size=EVAL_SAMPLE_SIZE,
)

print(f"\n[GRPO] seqeval F1 : {grpo_f1:.4f}")
print(f"\n[GRPO] Classification Report:\n{grpo_report}")

In [ ]:
from src.scripts.utils import predict_entities_batch

DEMO_TEXTS = [
    "Tratando-se de ação indenizatória ajuizada por pessoa incapaz , é obrigatória a intervenção do Ministério Público na condição de fiscal da ordem jurídica .",
    "O ministério público acatou a decisão do STF e pediu a suspensão do processo contra o ex-presidente Lula , com base na Lei 17.681/2017 .",
]

# Use the TTLlama EOS token (</s>) instead of the Qwen3-specific <|im_end|>
extra_tokens = [
    "PESSOA", "ORGANIZACAO", "LOCAL", "TEMPO",
    "LEGISLACAO", "JURISPRUDENCIA", ":", ";", "\n", "-",
    t.tokenizer.eos_token,
]

results = predict_entities_batch(DEMO_TEXTS, model, t.tokenizer, extra_tokens=extra_tokens)

for text, result in zip(DEMO_TEXTS, results):
    resposta_idx = result.find("Resposta:")
    answer = result[resposta_idx:] if resposta_idx != -1 else result
    print(f"INPUT : {text}")
    print(f"OUTPUT: {answer.strip()}")
    print()

In [ ]:
print("=" * 44)
print(f"{'Model':<10} {'seqeval F1':>12}  {'Delta':>10}")
print("=" * 44)
print(f"{'SFT':<10} {sft_f1:>12.4f}  {'(baseline)':>10}")
delta = grpo_f1 - sft_f1
sign  = "+" if delta >= 0 else ""
print(f"{'GRPO':<10} {grpo_f1:>12.4f}  {sign}{delta:>9.4f}")
print("=" * 44)